### Setup Data loader

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from notebooks.create_deepGP import ServiceGP, prepare_chained_data
import gpytorch
from agent.components.commons import ServiceFeatureMapping, ServiceType
from typing import List
import torch

from agent.components import RASK

torch.set_default_dtype(torch.float64)


In [2]:

class DynamicServiceChain(torch.nn.Module):
    def __init__(self, service_configs: List[ServiceFeatureMapping], num_inducing: int = 64):
        super().__init__()
        self.configs = service_configs
        self.gp_layers = torch.nn.ModuleList()
        self.likelihoods = torch.nn.ModuleList()

        for i, config in enumerate(service_configs):
            # The first service takes only raw features.
            # Subsequent services take raw features + 1 (previous service output).
            input_dims = len(config.feature_indices)
            if i > 0:
                input_dims += 1

            gp = ServiceGP(input_dims=input_dims, num_inducing=num_inducing)
            likelihood = gpytorch.likelihoods.GaussianLikelihood()

            self.gp_layers.append(gp)
            self.likelihoods.append(likelihood)

    def forward(self, x):
        dists = []
        last_output = None

        for i, gp in enumerate(self.gp_layers):
            # Extract raw features for this service
            indices = self.configs[i].feature_indices
            current_input = x[:, indices]

            # If not the first service, concat the sample from the previous GP
            if last_output is not None:
                current_input = torch.cat([current_input, last_output], dim=-1)

            dist = gp(current_input)
            dists.append(dist)

            # Reparameterization trick sample for the next layer's input
            last_output = dist.rsample().unsqueeze(-1)

        return tuple(dists)

In [3]:

raw_df = pd.read_csv("../statics/metrics_20_0.csv")
converted_df = RASK.preprocess_data(raw_df)

QR_MAP = ServiceFeatureMapping(ServiceType.QR, [0, 1])
CV_MAP = ServiceFeatureMapping(ServiceType.CV, [2, 3, 4])
PC_MAP = ServiceFeatureMapping(ServiceType.PC, [5, 6])

# TODO: n_quantiles (100) is greater than the total number of samples (30). n_quantiles is set to n_samples.
repetitions = 10
# chain_definition = ([QR_MAP])
chain_definition = ([QR_MAP] * repetitions) + ([CV_MAP] * repetitions) + ([PC_MAP] * repetitions)


train_loader, test_x, test_y, scaler_X = prepare_chained_data(converted_df, chain_definition, test_size=0.5)


/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (30). n_quantiles is set to n_samples.
  warnings.warn(
/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (30). n_quantiles is set to n_samples.
  warnings.warn(
/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (30). n_quantiles is set to n_samples.
  warnings.warn(
/home/boris-pgx/development/composable-autonomous-offerings/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_data.py:2885: UserWarning: n_quantiles (100) is greater than the total number of samples (30)

### Setup the Structure


In [4]:

model = DynamicServiceChain(chain_definition, num_inducing=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

# Dynamic MLL setup
mlls = [
    gpytorch.mlls.VariationalELBO(model.likelihoods[i], model.gp_layers[i], num_data=len(train_loader.dataset))
    for i in range(len(chain_definition))
]

# Training Loop
model.train()
for epoch in range(500):
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        # Returns a tuple of distributions: (qr_dist, cv_dist, pc_dist)
        distributions = model(x_batch)

        # Calculate sum of losses dynamically
        loss = sum([-mlls[i](distributions[i], y_batch[:, i]) for i in range(len(mlls))])

        loss.backward()
        optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")
    if epoch % 50 == 0:
        for i, likelihood in enumerate(model.likelihoods):
            print(f"Noise: {likelihood.noise.item():.4f}")

Epoch 0 | Loss: 38.0546
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Noise: 0.6958
Epoch 10 | Loss: 33.4731
Epoch 20 | Loss: 30.2368
Epoch 30 | Loss: 28.2415
Epoch 40 | Loss: 26.3175
Epoch 50 | Loss: 25.0496
Noise: 0.6211
Noise: 0.6004
Noise: 0.5961
Noise: 0.5923
Noise: 0.5855
Noise: 0.5782
Noise: 0.5782
Noise: 0.5696
Noise: 0.5677
Noise: 0.5675
Noise: 0.5678
Noise: 0.5677
Noise: 0.5683
Noise: 0.5669
Noise: 0.5689
Noise: 0.5663
Noise: 0.5673
Noise: 0.5662
Noise: 0.5669
Noise: 0.5673
Noise: 0.5673
Noise: 0.5656
Noise: 0.5639
Noise: 0.5641
Noise: 0.5632
Noise: 0.5650
Noise: 0.5651
Noise: 0.5649
Noise: 0.5632
Noise: 0.5632
Epoch 60 | 

In [5]:
model.eval()
all_samples = []
num_mc = 100  # Number of Monte Carlo simulations per data point
num_services = len(model.configs)

with torch.no_grad(), gpytorch.settings.cholesky_jitter(1e-3):
    for _ in range(num_mc):
        # 1. model(test_x) now returns a tuple of N distributions
        distributions = model(test_x)

        # 2. Sample from each distribution in the chain
        # We use a list comprehension to handle all N services dynamically
        samples = [dist.sample() for dist in distributions]

        # 3. Stack along the last dimension to keep (N_test, N_services)
        s = torch.stack(samples, dim=-1)
        all_samples.append(s)

    # 4. Stack all MC trials into a single NumPy array
    # Shape: (num_mc, num_test_samples, num_services)
    combined_samples = torch.stack(all_samples, dim=0).cpu().numpy()

print(f"Inference complete. Samples shape: {combined_samples.shape}")

Inference complete. Samples shape: (100, 3, 30)


In [6]:
import numpy as np

# --- MODEL VALIDATION ---
# 1. Get the mean prediction for each service across Monte Carlo samples
# combined_samples shape: (num_mc, N_test, num_services)
predicted_means = combined_samples.mean(axis=0)

# 2. Extract the ground truth (test_y)
actual_y = test_y.cpu().numpy()

# 3. Calculate Error for each service dynamically
# We iterate based on the number of services defined in the model
for i, config in enumerate(model.configs):
    # Calculate RMSE for the i-th service in the chain
    rmse = np.sqrt(np.mean((predicted_means[:, i] - actual_y[:, i]) ** 2))
    print(f"{config.service_type.value} Service RMSE: {rmse:.4f}")

elastic-workbench-qr-detector Service RMSE: 0.0799
elastic-workbench-qr-detector Service RMSE: 0.0839
elastic-workbench-qr-detector Service RMSE: 0.0662
elastic-workbench-qr-detector Service RMSE: 0.0418
elastic-workbench-qr-detector Service RMSE: 0.0252
elastic-workbench-qr-detector Service RMSE: 0.0249
elastic-workbench-qr-detector Service RMSE: 0.0369
elastic-workbench-qr-detector Service RMSE: 0.0490
elastic-workbench-qr-detector Service RMSE: 0.0487
elastic-workbench-qr-detector Service RMSE: 0.0448
elastic-workbench-cv-analyzer Service RMSE: 0.0601
elastic-workbench-cv-analyzer Service RMSE: 0.0800
elastic-workbench-cv-analyzer Service RMSE: 0.0616
elastic-workbench-cv-analyzer Service RMSE: 0.0721
elastic-workbench-cv-analyzer Service RMSE: 0.0713
elastic-workbench-cv-analyzer Service RMSE: 0.0703
elastic-workbench-cv-analyzer Service RMSE: 0.0686
elastic-workbench-cv-analyzer Service RMSE: 0.0690
elastic-workbench-cv-analyzer Service RMSE: 0.0603
elastic-workbench-cv-analyzer S